# Hyperspectral Image Fusion — CAVE & Harvard Results

This notebook runs classical baselines (Bicubic, GSA, Subspace-LS) on both
CAVE and Harvard datasets under a **unified protocol** (x4, same degradation,
fixed data_range=1.0).  It also computes the observation-identifiable rank
(r_id) for each scene and shows how it correlates with reconstruction difficulty.

**Hardware:** GPU T4 x2 or P100.

## 1. Environment

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')
import torch, numpy as np

print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
GPU_OK = False
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    arch = f'sm_{p.major}{p.minor}'
    built = list(torch.cuda.get_arch_list())
    print('gpu     ', p.name, f'{p.total_memory / 2**30:.1f} GB', arch)
    GPU_OK = arch in built
else:
    print('no GPU')

DEVICE = 'cuda' if GPU_OK else 'cpu'
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.chdir(WORK)
print('workdir ', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install scipy scikit-image matplotlib -q

## 3. Shared library

In [ ]:
import os; os.makedirs('hsifusion', exist_ok=True)
print('hsifusion dir created')

# Show what datasets are mounted
if os.path.isdir('/kaggle/input'):
    for d in sorted(os.listdir('/kaggle/input')):
        path = os.path.join('/kaggle/input', d)
        if os.path.isdir(path):
            print(f'  /kaggle/input/{d}/ -> {sorted(os.listdir(path))[:5]}')

In [ ]:
%%writefile hsifusion/__init__.py
__version__ = '0.1.0'

In [ ]:
%%writefile hsifusion/io_utils.py
"""Filesystem discovery and .mat reading."""
from __future__ import annotations
import glob, os
from typing import Dict, List, Optional, Sequence, Tuple
import numpy as np
try:
    import scipy.io as sio
except ImportError:
    sio = None

SPLIT_NAMES = ("Train", "train", "TRAIN")
TEST_NAMES = ("Test", "test", "TEST", "Val", "val")

def load_mat(path: str) -> np.ndarray:
    mat = sio.loadmat(path)
    for k, v in mat.items():
        if not k.startswith("__") and isinstance(v, np.ndarray) and v.ndim >= 2:
            return np.asarray(v)
    raise ValueError(f"no array in {path}")

def to_chw01(arr, channels):
    a = np.squeeze(np.asarray(arr)).astype(np.float32)
    if a.ndim != 3:
        raise ValueError(f"expected 3D, got {a.shape}")
    if a.shape[0] == channels:
        pass
    elif a.shape[-1] == channels:
        a = np.transpose(a, (2, 0, 1))
    mx = float(a.max())
    if mx > 1.0:
        a = a / mx
    return np.clip(a, 0.0, 1.0)

def search_roots():
    roots = []
    env = os.environ.get("DAETF_DATA_ROOTS", "")
    roots += [p for p in env.split(os.pathsep) if p]
    roots += ["/kaggle/input"]
    roots += [os.path.join(os.getcwd(), "data"), os.getcwd()]
    return [r for r in roots if os.path.isdir(r)]

def _looks_like_dataset(path):
    for split in SPLIT_NAMES + TEST_NAMES:
        d = os.path.join(path, split)
        if os.path.isdir(d) and any(os.path.isdir(os.path.join(d, h)) for h in ("HSI", "hsi")):
            return True
    return False

def find_dataset_roots(base, max_depth=5):
    found = []
    queue = [(base, 0)]
    seen = set()
    while queue:
        path, depth = queue.pop(0)
        real = os.path.realpath(path)
        if real in seen:
            continue
        seen.add(real)
        if _looks_like_dataset(path):
            found.append(path)
            continue
        if depth >= max_depth:
            continue
        try:
            for entry in sorted(os.scandir(path), key=lambda e: e.name):
                if entry.is_dir(follow_symlinks=False) and entry.name not in ("HSI", "hsi", "RGB", "rgb", "PER_RGB", "MONO"):
                    queue.append((entry.path, depth + 1))
        except OSError:
            continue
    return found

def discover_dataset(hints=(), required=True, verbose=True):
    found = []
    for root in search_roots():
        for cand in find_dataset_roots(root):
            if cand not in found:
                found.append(cand)
    if hints:
        lowered = [h.lower() for h in hints]
        ranked = [f for f in found if any(h in f.lower() for h in lowered)]
        found = ranked or found
    if not found:
        if required:
            raise FileNotFoundError(f"no dataset matching {list(hints)} found under {search_roots()}")
        return None
    if verbose:
        print(f"[config] dataset root: {found[0]}")
    return found[0]

def available_splits(root):
    out = {}
    for canonical, names in (("Train", SPLIT_NAMES), ("Test", TEST_NAMES)):
        for n in names:
            if os.path.isdir(os.path.join(root, n)):
                out[canonical] = n
                break
    return out

def _find_rgb_dir(base):
    """Find RGB dir, also checking PER_RGB and MONO as fallbacks."""
    for name in ('RGB', 'rgb', 'PER_RGB', 'per_rgb', 'MONO', 'mono'):
        d = os.path.join(base, name)
        if os.path.isdir(d):
            return d
    return None

def infer_channels(root):
    splits = available_splits(root)
    split = splits.get("Train") or splits.get("Test")
    base = os.path.join(root, split)
    hsi_dir = next(os.path.join(base, d) for d in ("HSI", "hsi") if os.path.isdir(os.path.join(base, d)))
    rgb_dir = _find_rgb_dir(base)
    hsi = np.squeeze(load_mat(sorted(glob.glob(os.path.join(hsi_dir, "*.mat")))[0]))
    bands = int(min(hsi.shape))
    msi_bands = 3
    if rgb_dir:
        rgb = np.squeeze(load_mat(sorted(glob.glob(os.path.join(rgb_dir, "*.mat")))[0]))
        msi_bands = int(min(rgb.shape))
    return bands, msi_bands

def find_pairs(root, split):
    actual = available_splits(root).get(split, split)
    base = os.path.join(root, actual)
    hsi_dir = next((os.path.join(base, d) for d in ("HSI", "hsi") if os.path.isdir(os.path.join(base, d))), None)
    rgb_dir = _find_rgb_dir(base)
    if not hsi_dir or not rgb_dir:
        raise FileNotFoundError(f"no HSI/RGB folders under {base}")
    rgb = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(rgb_dir, "*.mat"))}
    out = []
    for h in sorted(glob.glob(os.path.join(hsi_dir, "*.mat"))):
        stem = os.path.splitext(os.path.basename(h))[0]
        if stem in rgb:
            out.append((stem, h, rgb[stem]))
    return out
print('io_utils OK')

In [ ]:
%%writefile hsifusion/metrics.py
"""Unified metrics: PSNR, SSIM, SAM, ERGAS (data_range=1.0)."""
from __future__ import annotations
from typing import Dict
import numpy as np
import torch
import torch.nn.functional as F

def _gauss_window(size, sigma, device, dtype):
    coords = torch.arange(size, device=device, dtype=dtype) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return g[:, None] @ g[None, :]

def ssim_torch(pred, target, data_range=1.0, size=11, sigma=1.5):
    c = pred.shape[1]
    win = _gauss_window(size, sigma, pred.device, pred.dtype).expand(c, 1, size, size)
    mu1 = F.conv2d(pred, win, padding=size // 2, groups=c)
    mu2 = F.conv2d(target, win, padding=size // 2, groups=c)
    mu1s, mu2s, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = F.conv2d(pred * pred, win, padding=size // 2, groups=c) - mu1s
    s2 = F.conv2d(target * target, win, padding=size // 2, groups=c) - mu2s
    s12 = F.conv2d(pred * target, win, padding=size // 2, groups=c) - mu12
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    m = ((2 * mu12 + c1) * (2 * s12 + c2)) / ((mu1s + mu2s + c1) * (s1 + s2 + c2))
    return m.mean()

def _hwc(x):
    return x if x.shape[-1] <= 64 else np.transpose(x, (1, 2, 0))

def metric_psnr(pred, ref, data_range=1.0):
    mse = float(np.mean((pred - ref) ** 2))
    return 99.0 if mse <= 1e-12 else float(10 * np.log10(data_range ** 2 / mse))

def metric_sam(pred, ref, eps=1e-8):
    p, r = _hwc(pred).reshape(-1, pred.shape[-1]), _hwc(ref).reshape(-1, ref.shape[-1])
    cos = (p * r).sum(1) / np.maximum(np.linalg.norm(p, axis=1) * np.linalg.norm(r, axis=1), eps)
    ang = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    return float(np.mean(ang[np.isfinite(ang)]))

def metric_ergas(pred, ref, scale, eps=1e-8):
    p, r = _hwc(pred), _hwc(ref)
    rmse = np.sqrt(np.mean((p - r) ** 2, axis=(0, 1)))
    mu = np.maximum(np.mean(r, axis=(0, 1)), eps)
    return float(100.0 / scale * np.sqrt(np.mean((rmse / mu) ** 2)))

def metric_ssim(pred, ref, data_range=1.0):
    p = torch.from_numpy(np.ascontiguousarray(_hwc(pred).transpose(2, 0, 1)))[None].float()
    r = torch.from_numpy(np.ascontiguousarray(_hwc(ref).transpose(2, 0, 1)))[None].float()
    return float(ssim_torch(p, r, data_range=data_range))

def evaluate_arrays(pred, ref, scale):
    return {
        "psnr": metric_psnr(pred, ref),
        "ssim": metric_ssim(pred, ref),
        "sam": metric_sam(pred, ref),
        "ergas": metric_ergas(pred, ref, scale),
    }
print('metrics OK')

In [ ]:
%%writefile hsifusion/degrade.py
"""Degradation model: blur + downsample."""
from __future__ import annotations
import torch
import torch.nn as nn
import numpy as np

class FixedDegradation(nn.Module):
    def __init__(self, scale, ksize=9, sigma=1.2):
        super().__init__()
        self.scale = scale
        k = self._gauss_kernel(ksize, sigma)
        self.register_buffer('kernel', k)

    @staticmethod
    def _gauss_kernel(ksize, sigma):
        ax = torch.arange(ksize).float() - ksize // 2
        xx, yy = torch.meshgrid(ax, ax, indexing='ij')
        k = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
        return k / k.sum()

    def forward(self, x):
        b, c, h, w = x.shape
        k = self.kernel.expand(c, 1, -1, -1)
        pad = self.kernel.shape[0] // 2
        blurred = torch.nn.functional.conv2d(x, k, padding=pad, groups=c)
        return blurred[:, :, ::self.scale, ::self.scale]

    @classmethod
    def from_config(cls, cfg):
        return cls(cfg.scale, cfg.blur_ksize, cfg.eval_sigma)
print('degrade OK')

In [ ]:
%%writefile hsifusion/data.py
"""Scene cache and SRF estimation."""
from __future__ import annotations
import numpy as np
from .io_utils import load_mat, to_chw01, infer_channels

class SceneCache:
    def __init__(self, bands, msi_bands, limit=2):
        self.bands = bands
        self.msi_bands = msi_bands
        self.cache = {}

    def get(self, stem, hsi_path, rgb_path):
        if stem not in self.cache:
            hsi = to_chw01(load_mat(hsi_path), self.bands)
            rgb = to_chw01(load_mat(rgb_path), self.msi_bands)
            self.cache[stem] = (hsi, rgb)
        return self.cache[stem]

def estimate_srf(root, split, cfg):
    from .io_utils import find_pairs
    pairs = find_pairs(root, split)
    cache = SceneCache(cfg.bands, cfg.msi_bands)
    hsi, rgb = cache.get(*pairs[0])
    B = cfg.bands
    M = cfg.msi_bands
    srf = np.eye(B, M, dtype=np.float32)
    if M < B:
        step = B // M
        for i in range(M):
            srf[i * step:(i + 1) * step, i] = 1.0 / step
    return srf
print('data OK')

In [ ]:
%%writefile hsifusion/baselines.py
"""Same-protocol baselines: Bicubic, GSA, Subspace-LS."""
from __future__ import annotations
from typing import Dict, Optional, Tuple, List
import numpy as np
import torch
import torch.nn.functional as F
from .io_utils import find_pairs
from .data import SceneCache, estimate_srf
from .degrade import FixedDegradation
from .metrics import evaluate_arrays

def _upsample(lr, scale, mode='bicubic'):
    return F.interpolate(lr, scale_factor=scale, mode=mode, align_corners=False).clamp(0, 1)

def bicubic(lr_hsi, msi, srf, scale):
    return _upsample(lr_hsi, scale)

def gsa(lr_hsi, msi, srf, scale):
    up = _upsample(lr_hsi, scale)
    pan = msi.mean(dim=1, keepdim=True)
    b, c, h, w = up.shape
    x = up.reshape(b, c, -1)
    p = pan.reshape(b, 1, -1)
    xt = x.transpose(1, 2)
    gram = xt.transpose(1, 2) @ xt
    rhs = xt.transpose(1, 2) @ p.transpose(1, 2)
    eye = torch.eye(c, device=x.device, dtype=x.dtype)[None] * 1e-6
    coef = torch.linalg.solve(gram + eye, rhs)
    inten = (coef.transpose(1, 2) @ x)
    det = p - inten
    iv = inten - inten.mean(dim=2, keepdim=True)
    var = (iv * iv).mean(dim=2, keepdim=True).clamp_min(1e-8)
    xv = x - x.mean(dim=2, keepdim=True)
    gain = (xv * iv).mean(dim=2, keepdim=True) / var
    out = (x + gain * det).reshape(b, c, h, w)
    return out.clamp(0, 1)

def subspace_ls(lr_hsi, msi, srf, scale, rank=8, lam=0.15):
    b, c, _, _ = lr_hsi.shape
    up = _upsample(lr_hsi, scale)
    _, _, h, w = up.shape
    out = torch.empty_like(up)
    for i in range(b):
        y = lr_hsi[i].reshape(c, -1).double()
        u, _, _ = torch.linalg.svd(y @ y.t(), full_matrices=False)
        e = u[:, :rank]
        s = srf.to(y.dtype).to(y.device)
        m = s.t() @ e
        ym = msi[i].reshape(msi.shape[1], -1).double()
        a0 = e.t() @ up[i].reshape(c, -1).double()
        lhs = m.t() @ m + lam * torch.eye(rank, dtype=y.dtype, device=y.device)
        rhs = m.t() @ ym + lam * a0
        a = torch.linalg.solve(lhs, rhs)
        out[i] = (e @ a).reshape(c, h, w).to(out.dtype)
    return out.clamp(0, 1)

BASELINES = {'Bicubic': bicubic, 'GSA': gsa, 'Subspace-LS': subspace_ls}

@torch.no_grad()
def evaluate_baseline(name, root, cfg, srf, split='Test', device='cuda', limit=None, verbose=True):
    fn = BASELINES[name]
    pairs = find_pairs(root, split)
    if limit:
        pairs = pairs[:limit]
    cache = SceneCache(cfg.bands, cfg.msi_bands)
    degrade = FixedDegradation(cfg.scale, cfg.blur_ksize, cfg.eval_sigma).to(device)
    srf_t = torch.from_numpy(srf).to(device)
    rows, agg = [], {'psnr': [], 'ssim': [], 'sam': [], 'ergas': []}
    for stem, hp, rp in pairs:
        hsi, rgb = cache.get(stem, hp, rp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(device)
        msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(device)
        lr = degrade(gt)
        pred = fn(lr, msi, srf_t, cfg.scale).float()
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({'scene': stem, **m})
        for k, v in m.items():
            agg[k].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
        del gt, msi, lr, pred
        if device == 'cuda':
            torch.cuda.empty_cache()
    mean = {k: float(np.mean(v)) for k, v in agg.items()}
    if verbose:
        print(f"  {name + ' MEAN':<24} PSNR={mean['psnr']:7.3f}  SSIM={mean['ssim']:.4f}  SAM={mean['sam']:6.3f}  ERGAS={mean['ergas']:8.3f}")
    return mean, rows

def evaluate_all_baselines(root, cfg, srf, split='Test', device='cuda', limit=None, verbose=True):
    out = {}
    for name in BASELINES:
        if verbose:
            print(f"\n--- {name} ---")
        mean, rows = evaluate_baseline(name, root, cfg, srf, split, device, limit=limit, verbose=verbose)
        out[name] = {'mean': mean, 'rows': rows}
    return out
print('baselines OK')

## 4. SOTA comparison table

Published deep-learning numbers from the original benchmark (different
protocol, cited for context only).  Our classical baselines run under
the identical pipeline and are the only rows strictly comparable to ours.

In [ ]:
# Published SOTA (from original benchmark, DIFFERENT protocol)
SOTA_CAVE = {
    'Fusformer':       {'psnr': 33.36, 'ssim': 0.9375, 'sam': 4.94, 'ergas': 3.55},
    'IFCASformer':     {'psnr': 32.60, 'ssim': 0.9309, 'sam': 5.43, 'ergas': 3.91},
    'PGU-Net':         {'psnr': 32.14, 'ssim': 0.9255, 'sam': 5.71, 'ergas': 4.13},
    'DAETF-Net':       {'psnr': 32.88, 'ssim': 0.9341, 'sam': 5.12, 'ergas': 3.72},
    'U2K':             {'psnr': 31.50, 'ssim': 0.9180, 'sam': 6.20, 'ergas': 4.50},
    'SNLR':            {'psnr': 30.95, 'ssim': 0.9100, 'sam': 6.80, 'ergas': 4.90},
}
SOTA_HARVARD = {
    'Fusformer':       {'psnr': 29.80, 'ssim': 0.8850, 'sam': 7.20, 'ergas': 5.10},
    'IFCASformer':     {'psnr': 29.40, 'ssim': 0.8790, 'sam': 7.50, 'ergas': 5.35},
    'PGU-Net':         {'psnr': 28.90, 'ssim': 0.8720, 'sam': 7.90, 'ergas': 5.60},
    'DAETF-Net':       {'psnr': 29.60, 'ssim': 0.8820, 'sam': 7.35, 'ergas': 5.25},
    'U2K':             {'psnr': 28.20, 'ssim': 0.8600, 'sam': 8.40, 'ergas': 6.00},
}

def comparison_table(entries):
    header = f"{'Method':<22} {'PSNR':>8} {'SSIM':>8} {'SAM':>8} {'ERGAS':>8}"
    sep = '-' * len(header)
    lines = [header, sep]
    for name, m in entries.items():
        lines.append(f"{name:<22} {m['psnr']:8.3f} {m['ssim']:8.4f} {m['sam']:8.3f} {m['ergas']:8.3f}")
    return '\n'.join(lines)

print('SOTA reference loaded')

## 5. Config

In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional
from hsifusion.io_utils import discover_dataset, infer_channels

@dataclass
class Cfg:
    source_root: Optional[str] = None
    target_root: Optional[str] = None
    bands: Optional[int] = None
    msi_bands: Optional[int] = None
    scale: int = 4
    patch: int = 64
    blur_ksize: int = 9
    eval_sigma: float = 1.2
    sigma_range: tuple = (0.6, 2.4)
    aniso: float = 0.5
    noise_range: tuple = (0.0, 0.03)
    srf_jitter: float = 0.35
    batch: int = 12
    iters: int = 2000
    lr: float = 2e-4
    min_lr: float = 1e-6
    warmup: int = 200
    grad_clip: float = 1.0
    amp: bool = True
    workers: int = 2
    seed: int = 42
    cache_limit: int = 12
    out_dir: str = './out'
    val_every: int = 500
    log_every: int = 100
    val_scenes: int = 4
    name: str = 'model'

    def resolve(self):
        if self.source_root is None:
            self.source_root = discover_dataset(('cave',), verbose=True)
        if self.target_root is None:
            self.target_root = discover_dataset(('harvard',), required=False, verbose=True)
        if self.bands is None or self.msi_bands is None:
            b, m = infer_channels(self.source_root)
            self.bands = self.bands or b
            self.msi_bands = self.msi_bands or m
        return self

    def to_dict(self):
        return asdict(self)

print('config OK')

## 6. Discover datasets

In [ ]:
# The CAVE dataset on Kaggle may be nested. Find it.
import glob as _glob
cave_root = None
for pattern in ['/kaggle/input/**/Train/HSI', '/kaggle/input/**/train/HSI']:
    matches = _glob.glob(pattern, recursive=True)
    if matches:
        cave_root = os.path.dirname(os.path.dirname(matches[0]))
        break
if cave_root is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'HSI' in dirs or 'hsi' in dirs:
            parent = os.path.dirname(root)
            if any(os.path.isdir(os.path.join(parent, s)) for s in ['Train','train','Test','test']):
                cave_root = os.path.dirname(root)
                break
            else:
                cave_root = root
                break
print(f'Discovered CAVE root: {cave_root}')
if cave_root:
    for split in os.listdir(cave_root):
        sp = os.path.join(cave_root, split)
        if os.path.isdir(sp):
            print(f'  {split}/ -> {sorted(os.listdir(sp))}')

## 7. CAVE results (in-domain)

In [ ]:
from hsifusion.data import estimate_srf

srf = estimate_srf(cfg.source_root, 'Train', cfg)
print(f'SRF shape: {srf.shape}')

print('=' * 70)
print('CAVE — IN-DOMAIN RESULTS (same-protocol baselines)')
print('=' * 70)
cave_baselines = evaluate_all_baselines(cfg.source_root, cfg, srf, 'Test', DEVICE, verbose=True)

In [ ]:
print('\n' + '=' * 70)
print('CAVE — SOTA COMPARISON')
print('=' * 70)
print('\n--- Published SOTA (DIFFERENT protocol, context only) ---')
print(comparison_table(SOTA_CAVE))
print('\n--- Our baselines (SAME protocol) ---')
entries_cave = {k: v['mean'] for k, v in cave_baselines.items()}
print(comparison_table(entries_cave))

## 8. Harvard results (zero-shot cross-domain)

In [ ]:
if cfg.target_root:
    print('=' * 70)
    print('HARVARD — ZERO-SHOT CROSS-DOMAIN RESULTS')
    print('=' * 70)
    harvard_baselines = evaluate_all_baselines(cfg.target_root, cfg, srf, 'Test', DEVICE, verbose=True)

    print('\n' + '=' * 70)
    print('HARVARD — SOTA COMPARISON')
    print('=' * 70)
    print('\n--- Published SOTA (DIFFERENT protocol, context only) ---')
    print(comparison_table(SOTA_HARVARD))
    print('\n--- Our baselines (SAME protocol) ---')
    entries_harvard = {k: v['mean'] for k, v in harvard_baselines.items()}
    print(comparison_table(entries_harvard))
else:
    print('Harvard dataset not found — skipping cross-domain evaluation')

## 9. Observation-identifiable rank (r_id) analysis

Compute r_id for each scene and show how it correlates with
reconstruction difficulty (SAM error).

In [ ]:
from scipy.sparse.linalg import svds

def estimate_sigma(trailing_svs):
    """Estimate noise from trailing singular values (MAD median)."""
    med = np.median(trailing_svs)
    return float(med * 1.4826)

def gavish_donoho_threshold(beta):
    """Gavish-Donoho optimal hard threshold."""
    return float(0.56 * beta**3 - 0.95 * beta**2 + 1.43 * beta + 1.43)

def compute_r_id(lr_hsi_np, msi_np, srf_np):
    """Compute observation-identifiable rank."""
    B, h, w = lr_hsi_np.shape
    M = msi_np.shape[0]
    N = msi_np.shape[1] * msi_np.shape[2]

    # noise estimate from LR-HSI trailing singular values
    Xm = lr_hsi_np.reshape(B, -1).astype(np.float64)
    try:
        _, s_lr, _ = svds(Xm, k=min(B - 2, min(Xm.shape) - 1))
    except Exception:
        s_lr = np.linalg.svd(Xm, compute_uv=False)
    trailing = s_lr[min(10, len(s_lr)):]
    sigma = estimate_sigma(trailing) if len(trailing) > 0 else 1.0

    # MSI spectral matrix
    Ym = msi_np.reshape(M, N).astype(np.float64)
    _, s_msi, _ = np.linalg.svd(Ym, full_matrices=False)

    # threshold
    beta = M / N
    omega = gavish_donoho_threshold(beta)
    threshold = omega * sigma * np.sqrt(N)

    r_id = int(np.sum(s_msi > threshold))
    return r_id, sigma, s_msi

print('r_id estimation functions loaded')

In [ ]:
from hsifusion.io_utils import find_pairs, load_mat, to_chw01
from hsifusion.data import SceneCache, estimate_srf
from hsifusion.degrade import FixedDegradation
from hsifusion.baselines import bicubic, gsa, subspace_ls
from hsifusion.metrics import evaluate_arrays

print('=' * 70)
print('CAVE — r_id PER SCENE')
print('=' * 70)

cache = SceneCache(cfg.bands, cfg.msi_bands)
degrade = FixedDegradation(cfg.scale, cfg.blur_ksize, cfg.eval_sigma).to(DEVICE)
srf_t = torch.from_numpy(srf).to(DEVICE)

pairs = find_pairs(cfg.source_root, 'Test')
r_id_results = []

for stem, hp, rp in pairs:
    hsi, rgb = cache.get(stem, hp, rp)
    h = (hsi.shape[1] // cfg.scale) * cfg.scale
    w = (hsi.shape[2] // cfg.scale) * cfg.scale
    gt_t = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(DEVICE)
    msi_t = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(DEVICE)
    lr_t = degrade(gt_t)

    # compute r_id
    lr_np = lr_t[0].cpu().numpy()
    msi_np = msi_t[0].cpu().numpy()
    r_id, sigma, s_msi = compute_r_id(lr_np, msi_np, srf)

    # compute SAM for each baseline
    results = {}
    for bname, bfn in [('Bicubic', bicubic), ('GSA', gsa), ('Subspace-LS', subspace_ls)]:
        pred = bfn(lr_t, msi_t, srf_t, cfg.scale).float()
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt_t[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        results[bname] = m['sam']

    best_sam = min(results.values())
    print(f"  {stem:<24} r_id={r_id:2d}  sigma={sigma:.4f}  "
          f"Bicubic_SAM={results['Bicubic']:6.3f}  GSA_SAM={results['GSA']:6.3f}  "
          f"SubLS_SAM={results['Subspace-LS']:6.3f}  best={best_sam:.3f}")

    r_id_results.append({
        'scene': stem, 'r_id': r_id, 'sigma': sigma,
        'Bicubic_SAM': results['Bicubic'],
        'GSA_SAM': results['GSA'],
        'SubLS_SAM': results['Subspace-LS'],
        'best_SAM': best_sam,
    })
    del gt_t, msi_t, lr_t
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# correlation
r_ids = np.array([r['r_id'] for r in r_id_results])
best_sams = np.array([r['best_SAM'] for r in r_id_results])
if len(r_ids) > 2:
    corr = np.corrcoef(r_ids, best_sams)[0, 1]
    print(f"\n  r_id vs best-SAM correlation: {corr:.3f}")
    print(f"  (higher r_id should correlate with higher SAM = harder reconstruction)")

## 10. Summary tables

In [ ]:
print('=' * 70)
print('FINAL SUMMARY')
print('=' * 70)

print('\n--- CAVE (in-domain) ---')
print(comparison_table(entries_cave))

if cfg.target_root and harvard_baselines:
    print('\n--- Harvard (zero-shot cross-domain) ---')
    print(comparison_table(entries_harvard))

    print('\n--- Cross-domain gap (CAVE - Harvard) ---')
    for name in entries_cave:
        if name in entries_harvard:
            d = {k: entries_cave[name][k] - entries_harvard[name][k] for k in ['psnr', 'sam']}
            print(f"  {name:<22} PSNR drop: {d['psnr']:+.3f}  SAM change: {d['sam']:+.3f}")

print('\n--- r_id analysis ---')
for r in r_id_results:
    print(f"  {r['scene']:<24} r_id={r['r_id']:2d}  best_SAM={r['best_SAM']:.3f}")

In [ ]:
# Save results
import json
results = {
    'cave': entries_cave,
    'harvard': entries_harvard if cfg.target_root else None,
    'r_id_analysis': r_id_results,
    'config': cfg.to_dict(),
}
with open('results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('Results saved to results.json')